# Notebook #4: Feature Correlations
We examine how the article-level features relate to each other, as a diagnostic for collinearity among the regression predictors (each row of the feature matrix is an article, features are columns).Continuous features are transformed as described in notebook 03 (publication age and number of authors are $log$-transformed on functional and distributional grounds respectively; venue impact stays on its natural scale) before this notebook uses them.
<br><br>
We compute pairwise correlations between features and the variance inflation factors (VIF) for each feature. Features with high VIF values (e.g., > 5 or 10) may indicate multicollinearity and may need to be removed or combined in subsequent analyses.
<br><br>
The final feature matrix includes the following features (columns) for each article (row):
- Is Sharing Data (binary)
- Sharing Class (ordinal)
- Has US Author (binary)
- Is Open Access (binary)
- Has Preprint (binary)
- Venue Impact (continuous)
- log(Weeks Since Pub.) (continuous)
- log(Number of Authors) (continuous)


In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
import plotly.graph_objects as go
import plotly.io as pio

from helpers.config import *
from helpers import dataset
from helpers.plotting import save_figure
from helpers.stats import significance_class

pio.renderers.default = "browser"


## Setup
The analytic sample is rebuilt (or loaded from the parquet cache) by `helpers.dataset.load_or_build()`, so this notebook runs standalone from a cold kernel.

In [2]:
combined, FEATURES_DF = dataset.load_or_build()
FEATURES_DF.head()

Loaded cached dataset from C:\Users\nirjo\Documents\University\PhD\Projects\eye_movement_data_sharing\data_store (234 rows)


,Is Sharing Data,Sharing Class,Has US Author,Is Open Access,Has Preprint,Venue Impact,log(Weeks Since Pub.),log(Number of Authors)
0,1,FIXATION,0,1,1,0.429831,5.653992,1.098612
2,0,NONE,0,0,0,1.419355,5.914661,1.609438
3,1,FIXATION,0,1,0,0.429831,5.481234,1.098612
4,1,FIXATION,0,1,0,3.520000,5.723585,1.098612
5,1,TRIAL,0,1,1,3.520000,5.459586,1.386294


### Pairwise Correlations
We calculate the pairwise correlations between article features, along with matching statistical test for significance, based on the types of the two features being compared:
- Binary-Binary: Pearson's `phi` coefficient (equivalent to Pearson r on 0/1 data) with Chi-square or Fisher's exact test for significance.
- Binary-Continuous: Point-biserial correlation with t-test (with $dof=N-1$) for significance.
- Continuous-Continuous: Pearson correlation with t-test for significance.

In this analysis, we exclude the `Sharing Class` feature, which is an ordinal refinement of `Is Sharing Data` and is therefore collinear by construction. We only include `Is Sharing Data` in the correlation matrix and VIF calculations.
<br><br>
To Visualize the results, we generate a heatmap of the pairwise correlation coefficients, with the upper-triangle of the matrix filled and thediagonal and lower-triangle left blank (NaN). We do not include the p-values in the heatmap, as this is a diagnostic for collinearity among the regression predictors, and only the magnitude of each coefficient matters (still, the uncorrected p-values are retained in the `pairs` dataframe for inspection only).

In [3]:
def pairwise_correlation(x, y, x_is_binary, y_is_binary):
    """Coefficient + p-value using the measure/test matched to the pair's variable types."""
    paired = pd.concat([x, y], axis=1).dropna()
    a, b = paired.iloc[:, 0], paired.iloc[:, 1]
    if x_is_binary and y_is_binary:                      # binary-binary -> phi + chi-square / Fisher
        table = pd.crosstab(a, b)
        _, p, _, expected = stats.chi2_contingency(table, correction=False)
        if (expected < 5).any():
            _, p = stats.fisher_exact(table)
        coefficient = np.corrcoef(a, b)[0, 1]            # phi == Pearson r on 0/1 data
        method = "phi (chi2/Fisher)"
    elif x_is_binary or y_is_binary:                     # binary-continuous -> point-biserial
        binary, continuous = (a, b) if x_is_binary else (b, a)
        coefficient, p = stats.pointbiserialr(binary, continuous)
        method = "point-biserial"
    else:                                                # continuous-continuous -> Pearson
        coefficient, p = stats.pearsonr(a, b)
        method = "pearson"
    return coefficient, p, method


In [4]:
CORRELATION_FEATURES = [col for col in FEATURES_DF.columns if col != "Sharing Class"]
correlation_df = FEATURES_DF[CORRELATION_FEATURES]
feature_names = list(correlation_df.columns)
n_features = len(feature_names)
coef_matrix = np.full((n_features, n_features), np.nan)
np.fill_diagonal(coef_matrix, 1.0)
pair_rows = []
for i in range(n_features):
    for j in range(i + 1, n_features):
        xi, xj = feature_names[i], feature_names[j]
        r, p, method = pairwise_correlation(
            correlation_df[xi], correlation_df[xj], xi in BINARY_FEATURES, xj in BINARY_FEATURES
        )
        coef_matrix[i, j] = coef_matrix[j, i] = r
        pair_rows.append({"i": i, "j": j, "X": xi, "Y": xj, "method": method, "r": r, "p_unc": p})

# NOTE: we calculate significance for these correlations but don't render in figures or report in
# the main article. This matrix is a *diagnostic* for collinearity among the regression predictors,
# so only the magnitude of each coefficient matters (see also the VIF table below).
pairs = pd.DataFrame(pair_rows)
pairs["p_fdr"] = multipletests(pairs["p_unc"], method="fdr_bh")[1]
pairs["sig"] = pairs["p_fdr"].map(significance_class)
pairs[["X", "Y", "method", "r", "p_unc", "p_fdr", "sig"]].round(4)

,X,Y,method,r,p_unc,p_fdr,sig
0,Is Sharing Data,Has US Author,phi (chi2/Fisher),-0.0519,0.4272,0.6849,n.s.
1,Is Sharing Data,Is Open Access,phi (chi2/Fisher),0.1254,0.0550,0.2434,n.s.
2,Is Sharing Data,Has Preprint,phi (chi2/Fisher),0.2908,0.0000,0.0002,***
3,Is Sharing Data,Venue Impact,point-biserial,0.2044,0.0017,0.0117,*
4,Is Sharing Data,log(Weeks Since Pub.),point-biserial,-0.2131,0.0010,0.0109,*
5,Is Sharing Data,log(Number of Authors),point-biserial,-0.0388,0.5545,0.6849,n.s.
6,Has US Author,Is Open Access,phi (chi2/Fisher),-0.0289,0.6582,0.7183,n.s.
7,Has US Author,Has Preprint,phi (chi2/Fisher),-0.0407,0.5337,0.6849,n.s.
8,Has US Author,Venue Impact,point-biserial,0.0464,0.4796,0.6849,n.s.
9,Has US Author,log(Weeks Since Pub.),point-biserial,-0.0432,0.5111,0.6849,n.s.


In [8]:
# N x N heatmap showing the diagonal and the upper triangle; the lower triangle is the mirror
# image and carries no extra information, so it is left blank (NaN -> uncolored). Keeping the
# diagonal (self-correlations, r = 1) means every row and column has at least one populated cell.
z_color = coef_matrix.copy()
z_color[np.tril_indices(n_features, k=-1)] = np.nan     # blank the STRICT lower triangle
axis_labels = [feat.replace("log", "<i>log</i>") for feat in feature_names]

corr_fig = go.Figure(go.Heatmap(
    z=z_color, x=axis_labels, y=axis_labels,
    colorscale="RdBu", zmid=0, zmin=-1, zmax=1, xgap=2, ygap=2,
    colorbar=dict(
        title=dict(text="<i>r</i>", font=AXIS_TITLE_FONT),
        tickfont=AXIS_TICK_FONT, thickness=14, len=0.85,
    ),
    hovertemplate="%{y} \u00d7 %{x}<extra></extra>",
))

# a heatmap trace has a single text colour, so the cell values are annotations instead: white
# over the dark red/blue of strong correlations (|r| > 0.8, diagonal included), black elsewhere
STRONG_CORRELATION = 0.8
for i in range(n_features):
    for j in range(i, n_features):                      # start at i to include the diagonal
        is_strong = abs(coef_matrix[i, j]) > STRONG_CORRELATION
        corr_fig.add_annotation(
            x=axis_labels[j], y=axis_labels[i], xref="x", yref="y",
            text=f"{coef_matrix[i, j]:.2f}", showarrow=False,
            font={**AXIS_TICK_FONT, "color": "white" if is_strong else "black"},
        )

# update layout and styling
corr_fig.update_yaxes(autorange="reversed", tickfont=AXIS_TICK_FONT)
corr_fig.update_xaxes(side="top", tickangle=-35, tickfont=AXIS_TICK_FONT)
# `mirror=True` repeats each axis line on the opposite side, closing the box on all four edges
corr_fig.update_xaxes(showline=True, linewidth=1, linecolor="black", mirror=True)
corr_fig.update_yaxes(showline=True, linewidth=1, linecolor="black", mirror=True)
corr_fig.update_layout(
    width=720, height=600,
    # `yref="container"` measures y against the whole figure rather than the plotting area, so the
    # title sits at a predictable offset from the top edge; `margin.t` then reserves room for both
    # the title and the angled x tick labels drawn beneath it.
    title=dict(
        text="<b>Pairwise Feature Correlations</b>", font=TITLE_FONT,
        x=0.5, xanchor="center", y=0.97, yanchor="top", yref="container",
    ),
    margin=dict(t=160, b=10, l=10, r=10),
    template="simple_white",
)

if False:
    corr_fig.show()

In [6]:
# flip to True to export this figure into `output/`
if False:
    save_figure(corr_fig, "corr_fig.png", width=720, height=600)

### Variance Inflation Factor
The variance inflation factor ([VIF](https://en.wikipedia.org/wiki/Variance_inflation_factor)) measures multi-collinearity between regression predictors.<br>
If predictor $X_i$ has a $VIF(X_i) = 1$ - it indicates that the predictor is completely independent of **all** other predictors in the model. Conversely, high values, e.g. $VIF(X_i) > 5$ indicate large multi-collinearity in the model, suggesting that some predictors should be excluded or pre-processed\* before adding them to the model.<br>
(\*pre processing predictors = recentering, normalizing, or combining two or more variables to a single one)

In [7]:
vif_input = sm.add_constant(correlation_df)
vif = pd.DataFrame({
    "variable": vif_input.columns,
    "VIF": [variance_inflation_factor(vif_input.values, i).round(4) for i in range(vif_input.shape[1])]
})
display(vif)

,variable,VIF
0,const,474.5634
1,Is Sharing Data,1.2053
2,Has US Author,1.0125
3,Is Open Access,1.0477
4,Has Preprint,1.1051
5,Venue Impact,1.0742
6,log(Weeks Since Pub.),1.0693
7,log(Number of Authors),1.0376
